In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier




In [3]:
df = pd.read_csv('telco_customer_data_v2.csv')

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,CUST00001,Male,0,No,Yes,3.0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,68.61,205.83,Yes
1,CUST00002,Male,1,Yes,No,2.0,Yes,Yes,DSL,No,...,No internet service,Yes,NaN,No,One year,Yes,Bank transfer (automatic),23.15,46.3,No
2,CUST00003,Female,No,No,No,42.0,Yes,Yes,DSL,No,...,No,NaN,Yes,Yes,Month-to-month,No,Electronic check,42.63,1790.46,Yes
3,CUST00004,Female,0,No,Yes,40.0,Yes,Yes,Fiber optic,No,...,Yes,No,No,No internet service,Month-to-month,No,Electronic check,75.04,3001.6,No
4,CUST00005,Male,Yes,Yes,Yes,17.0,Yes,NaN,Fiber optic,Yes,...,Yes,No,No internet service,No,Two year,Yes,Electronic check,22.38,380.46,Yes


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        70000 non-null  object 
 1   gender            69252 non-null  object 
 2   SeniorCitizen     69341 non-null  object 
 3   Partner           66470 non-null  object 
 4   Dependents        66435 non-null  object 
 5   tenure            69433 non-null  float64
 6   PhoneService      70000 non-null  object 
 7   MultipleLines     68132 non-null  object 
 8   InternetService   70000 non-null  object 
 9   OnlineSecurity    67078 non-null  object 
 10  OnlineBackup      67253 non-null  object 
 11  DeviceProtection  67106 non-null  object 
 12  TechSupport       67267 non-null  object 
 13  StreamingTV       67173 non-null  object 
 14  StreamingMovies   67215 non-null  object 
 15  Contract          70000 non-null  object 
 16  PaperlessBilling  70000 non-null  object

In [6]:
df.shape

(70000, 21)

In [7]:
df.describe()

,tenure,MonthlyCharges
count,69433.000000,69612.000000
mean,30.516858,60.588548
std,89.873767,111.509588
min,-10.000000,18.000000
25%,10.000000,29.670000
50%,20.000000,41.190000
75%,35.000000,63.882500
max,999.000000,1499.770000


In [8]:
missing_table = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percent": df.isnull().mean() * 100
})

missing_table = missing_table[missing_table["Missing_Count"] > 0]
missing_table = missing_table.sort_values("Missing_Percent", ascending=False)

print(missing_table)


                  Missing_Count  Missing_Percent
PaymentMethod              3569         5.098571
Dependents                 3565         5.092857
Partner                    3530         5.042857
OnlineSecurity             2922         4.174286
DeviceProtection           2894         4.134286
StreamingTV                2827         4.038571
StreamingMovies            2785         3.978571
OnlineBackup               2747         3.924286
TechSupport                2733         3.904286
MultipleLines              1868         2.668571
TotalCharges               1062         1.517143
gender                      748         1.068571
SeniorCitizen               659         0.941429
tenure                      567         0.810000
MonthlyCharges              388         0.554286


In [9]:
cat_cols = df.select_dtypes(include=['object']).columns
cat_cols = [col for col in cat_cols if "id" not in col.lower()]


In [10]:
summary = pd.DataFrame({
    "Unique_Values": df[cat_cols].nunique(),
    "Mode": df[cat_cols].mode().iloc[0],
    "Mode_Frequency": df[cat_cols].apply(lambda x: x.value_counts().iloc[0]),
    "Missing_%": df[cat_cols].isnull().mean() * 100
}).round(2)

print(summary)


                  Unique_Values              Mode  Mode_Frequency  Missing_%
gender                        7            Female           34110       1.07
SeniorCitizen                 5                 0           52405       0.94
Partner                       2                No           38344       5.04
Dependents                    2                No           45375       5.09
PhoneService                  2               Yes           63011       0.00
MultipleLines                 3               Yes           34681       2.67
InternetService               3       Fiber optic           38502       0.00
OnlineSecurity                5                No           39195       4.17
OnlineBackup                  5                No           39308       3.92
DeviceProtection              5                No           39199       4.13
TechSupport                   5                No           39401       3.90
StreamingTV                   5                No           39198       4.04

In [11]:
df.drop(columns='customerID',inplace=True)

In [12]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Male,0,No,Yes,3.0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,68.61,205.83,Yes
1,Male,1,Yes,No,2.0,Yes,Yes,DSL,No,No,No internet service,Yes,NaN,No,One year,Yes,Bank transfer (automatic),23.15,46.3,No
2,Female,No,No,No,42.0,Yes,Yes,DSL,No,Yes,No,NaN,Yes,Yes,Month-to-month,No,Electronic check,42.63,1790.46,Yes
3,Female,0,No,Yes,40.0,Yes,Yes,Fiber optic,No,No,Yes,No,No,No internet service,Month-to-month,No,Electronic check,75.04,3001.6,No
4,Male,Yes,Yes,Yes,17.0,Yes,NaN,Fiber optic,Yes,No,Yes,No,No internet service,No,Two year,Yes,Electronic check,22.38,380.46,Yes


In [13]:
df['gender'].value_counts()

gender
Female    34110
Male      33252
Man         500
male        374
FEMALE      351
m           339
f           326
Name: count, dtype: int64

In [14]:
df['gender'] = df['gender'].replace({
    "m": "Male",
    "male": "Male",
    "Male": "Male",
    "Man": "Male",
    "f": "Female",
    "female": "Female",
    "FEMALE": "Female",
    "Woman": "Female",
    "M": "Male",
    "F": "Female"
})

In [15]:
df['gender'].value_counts()

gender
Female    34787
Male      34465
Name: count, dtype: int64

In [16]:
df['SeniorCitizen'] = df['SeniorCitizen'].replace({
    "Yes": "Yes",
    "No": "No",
    "not senior": "No",
    "1": "Yes",
    "0": "No"
})

In [17]:
df['SeniorCitizen'].value_counts()

SeniorCitizen
No     55394
Yes    13947
Name: count, dtype: int64

In [18]:
df['Partner'].value_counts()

Partner
No     38344
Yes    28126
Name: count, dtype: int64

In [19]:
df['Dependents'].value_counts()

Dependents
No     45375
Yes    21060
Name: count, dtype: int64

In [20]:
df['PhoneService'].value_counts()

PhoneService
Yes    63011
No      6989
Name: count, dtype: int64

In [21]:
df['MultipleLines'].value_counts()

MultipleLines
Yes                 34681
No                  25192
No phone service     8259
Name: count, dtype: int64

In [22]:
df['MultipleLines'] = df['MultipleLines'].replace({
    'No phone service': 'No'
})

In [23]:
df['MultipleLines'].value_counts()

MultipleLines
Yes    34681
No     33451
Name: count, dtype: int64

In [24]:
df['InternetService'].value_counts()

InternetService
Fiber optic    38502
DSL            17594
No             13904
Name: count, dtype: int64

In [25]:
df['OnlineSecurity'].value_counts()

OnlineSecurity
No                     39195
No internet service    16672
Yes                    11202
True                       6
Y                          3
Name: count, dtype: int64

In [26]:
df['OnlineSecurity'] = df['OnlineSecurity'].replace({
    'No internet service': 'No',
    'True': 'Yes',
    'Y': 'Yes'
    
    
})

In [27]:
df['OnlineSecurity'].value_counts()

OnlineSecurity
No     55867
Yes    11211
Name: count, dtype: int64

In [28]:
df['OnlineBackup'].value_counts()

OnlineBackup
No                     39308
No internet service    16691
Yes                    11237
Y                         10
True                       7
Name: count, dtype: int64

In [29]:
df['OnlineBackup'] = df['OnlineBackup'].replace({
    'No internet service' : 'No',
    'Y':'Yes',
    'True':'Yes'
})

In [30]:
df['OnlineBackup'].value_counts()

OnlineBackup
No     55999
Yes    11254
Name: count, dtype: int64

In [31]:
df['DeviceProtection'].value_counts()

DeviceProtection
No                     39199
No internet service    16623
Yes                    11277
Y                          6
True                       1
Name: count, dtype: int64

In [32]:
df['DeviceProtection'] = df['DeviceProtection'].replace({
    'No internet service' : 'No',
    'Y':'Yes',
    'True':'Yes'
})

In [33]:
df['DeviceProtection'].value_counts()

DeviceProtection
No     55822
Yes    11284
Name: count, dtype: int64

In [34]:
df['TechSupport'].value_counts()

TechSupport
No                     39401
No internet service    16616
Yes                    11239
Y                          7
True                       4
Name: count, dtype: int64

In [35]:
df['TechSupport'] = df['TechSupport'].replace({
    'No internet service' : 'No',
    'Y':'Yes',
    'True':'Yes'
})

In [36]:
df['TechSupport'].value_counts()

TechSupport
No     56017
Yes    11250
Name: count, dtype: int64

In [37]:
df['StreamingTV'].value_counts()

StreamingTV
No                     39198
No internet service    16610
Yes                    11355
Y                          5
True                       5
Name: count, dtype: int64

In [38]:
df['StreamingTV'] = df['StreamingTV'].replace({
    'No internet service' : 'No',
    'Y':'Yes',
    'True':'Yes'
})

In [39]:
df['StreamingTV'].value_counts()

StreamingTV
No     55808
Yes    11365
Name: count, dtype: int64

In [40]:
df['StreamingMovies'].value_counts()

StreamingMovies
No                     39189
No internet service    16744
Yes                    11271
Y                          7
True                       4
Name: count, dtype: int64

In [41]:
df['StreamingMovies'] = df['StreamingMovies'].replace({
    'No internet service' : 'No',
    'Y':'Yes',
    'True':'Yes'
})

In [42]:
df['StreamingMovies'].value_counts()

StreamingMovies
No     55933
Yes    11282
Name: count, dtype: int64

In [43]:
df['Contract'].value_counts()

Contract
Month-to-month    41845
One year          14015
Two year          13974
M-M                 166
Name: count, dtype: int64

In [44]:
df['Contract'] =df['Contract'].replace({
    'M-M':'Month-to-month'
})

In [45]:
df['Contract'].value_counts()

Contract
Month-to-month    42011
One year          14015
Two year          13974
Name: count, dtype: int64

In [46]:
df['PaperlessBilling'].value_counts()

PaperlessBilling
Yes    42011
No     27989
Name: count, dtype: int64

In [47]:
df['PaymentMethod'].value_counts()

PaymentMethod
Electronic check             27841
Bank transfer (automatic)    14102
Mailed check                 13963
Credit card (automatic)      10400
BANK TRANSFER                  125
Name: count, dtype: int64

In [48]:
df['PaymentMethod'] = df['PaymentMethod'].replace({
    'BANK TRANSFER': 'Bank transfer (automatic)'
    
})

In [49]:
df['PaymentMethod'].value_counts()


PaymentMethod
Electronic check             27841
Bank transfer (automatic)    14227
Mailed check                 13963
Credit card (automatic)      10400
Name: count, dtype: int64

In [50]:
x = df.drop('Churn',axis=1)
y = df['Churn']

In [51]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.20 , random_state=42)

In [52]:
x_train.shape

(56000, 19)

In [53]:
x_train.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
47339,Female,No,No,No,44.0,Yes,No,Fiber optic,No,Yes,No,Yes,Yes,No,One year,Yes,Bank transfer (automatic),25.73,1132.12
67456,Male,Yes,No,No,24.0,Yes,No,No,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,57.84,1388.16
12308,Female,Yes,Yes,No,48.0,Yes,Yes,DSL,No,No,No,No,Yes,No,Two year,Yes,Mailed check,26.30,1262.4
32557,Male,No,No,No,21.0,No,No,DSL,No,Yes,No,NaN,No,No,One year,Yes,Mailed check,23.54,494.34
664,Male,No,No,No,17.0,Yes,No,Fiber optic,No,No,No,No,Yes,Yes,Two year,Yes,Mailed check,42.84,728.28


In [54]:
si = SimpleImputer()

In [55]:
num_cols = x.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = x.select_dtypes(include=['object']).columns.tolist()


In [56]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean'))
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [57]:

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Final Pipeline
model = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

model.fit(x_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer())]),
                                                  ['tenure', 'MonthlyCharges']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['gender', 'SeniorCitizen',
                                                   'Partner', 'Dependents',
                                                   'PhoneService',
                                                   'MultipleLines',
                                                   'InternetService',
                                                   'OnlineSecurity',
                                                   'OnlineBackup',
                                                   'DeviceProtection',
                                                   'TechSupport', 'StreamingTV',
                                                   'StreamingMovies',
                                                   'Contract',
                                                   'PaperlessBilling',
                                                   'PaymentMethod',
                                                   'TotalCharges'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [58]:
y_pred = model.predict(x_test)

In [59]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)



Accuracy: 0.7678571428571429
